# 利用卡爾曼濾波器分解之混合預測模型的重抽樣分析與 Bootstrap 推論

## 一、研究說明

這份研究使用 **resampling methods** 來評估一個由 **Kalman filter 分解、ARIMA 與 ANN** 所組成的 hybrid forecasting model。整體流程是先建立模擬時間序列資料，再將觀測訊號分成低頻與高頻兩部分，分別建模後加總成最終預測。接著，我利用 Monte Carlo 重複模擬來估計 estimator 的 **bias、variance 與 MSE**，並進一步使用 **bootstrap methods** 建構 confidence intervals 與進行 hypothesis testing。

本作業主要完成以下幾件事：

1. 建立包含低頻線性成分、高頻非線性成分與 measurement noise 的模擬資料。
2. 使用 Kalman filter 將觀測序列分解為低頻部分與高頻部分。
3. 對低頻部分使用 ARIMA，對高頻部分使用 ANN 做遞迴預測。
4. 透過 Monte Carlo 模擬估計 forecast estimator 的 bias、variance 與 MSE。
5. 使用 bootstrap resampling 建構預測區間，並比較不同區間方法的表現。
6. 對 horizon 1 的平均預測誤差進行 bootstrap hypothesis testing。


## 二、研究動機

在時間序列分析中，若只看單一次點預測，通常無法完整描述模型的表現。除了預測值是否接近真值之外，我們也關心估計量是否穩定、誤差大小如何，以及模型不確定性應如何量化。這些問題在實務上都非常重要，因為預測結果往往伴隨隨機波動，而不只是單一固定值。

**resampling methods** 提供了一個很有用的工具，讓我們能在不完全依賴理論推導的情況下，透過樣本重抽方式估計統計量的變異、建構 confidence intervals，甚至進行 hypothesis testing。因此我的目標不只是建立一個 hybrid forecast model，而是進一步用 resampling 方法評估其統計性質，回答以下問題：

- estimator 的 bias 是否接近 0？
- estimator 的 variance 與 MSE 大小如何？
- bootstrap confidence interval 的 coverage 與寬度表現如何？
- 平均預測誤差是否顯著不等於 0？


## 三、模型設定

模擬資料的組成如下：

- 一個低頻線性成分
- 一個高頻非線性成分
- 一個 measurement noise

觀測模型可寫成：

$Y_t = L_t + H_t + \varepsilon_t$

其中：

- $L_t$ 表示低頻線性訊號
- $H_t$ 表示高頻非線性訊號
- $\varepsilon_t$ 表示觀測誤差

在模擬設計中，低頻部分使用 AR(1)-style process 生成，高頻部分則透過非線性遞迴關係生成。由於觀測資料是混合訊號，因此先使用簡化的 Kalman filter 將觀測值分解為平滑的低頻部分與殘差型的高頻部分，再分別對兩者建模。


## 四、模型設計

forecasting model 採用 hybrid 架構，分成以下幾個步驟：

1. **Kalman filter decomposition**  
   先將觀測資料分解成 estimated low-frequency component 與 estimated high-frequency component。

2. **Low-frequency forecasting**  
   對 estimated low component 建立 ARIMA model，並進行多步預測。

3. **High-frequency forecasting**  
   對 estimated high component 建立 ANN model，使用 lagged windows 作為輸入，並透過 recursive forecasting 產生未來預測值。

4. **Hybrid forecast**  
   將 low-frequency forecast 與 high-frequency forecast 相加，得到最終預測值。

5. **Rolling out-of-sample residual resampling**  
   為了避免只依賴 in-sample residuals，我使用 rolling window 的方式，在訓練樣本內部建立多個 out-of-sample prediction blocks。對每個 block，我先以前段資料建立 hybrid forecast model，再將實際值與預測值的差定義為 out-of-sample residuals。最後從這些 rolling residuals 中進行 resampling with replacement，建構 bootstrap forecast paths，並據此建立 confidence intervals。

6. **Monte Carlo evaluation**  
   重複進行整個資料生成與預測流程，以估計 estimator 的 bias、variance 與 MSE。

Monte Carlo simulation 用來評估 hybrid estimator 的表現。每一次模擬的流程如下：

1. 生成一組包含低頻、高頻與 noise 的時間序列資料。
2. 使用前段訓練資料建立 hybrid forecast model。
3. 對未來 horizon 進行預測。
4. 在訓練區間內額外使用 rolling window 方式建立 out-of-sample residual pool。
5. 從 rolling residual pool 中進行 bootstrap resampling，以建構 confidence intervals。
6. 比較預測值與真實 clean signal，並計算 bias、variance、MSE 與 interval performance。
7. 重複上述程序多次，彙整整體結果。

在統計量定義上，令第 $r$ 次模擬在 horizon $h$ 的預測值為 $\hat{\theta}_r(h)$，真值為 $\theta_r(h)$，則：

$Bias(h) = \frac{1}{R}\sum_{r=1}^{R}\left(\hat{\theta}_r(h)-\theta_r(h)\right)$

$Var(h) = \mathrm{Var}\left(\hat{\theta}_r(h)\right)$

$MSE(h) = \frac{1}{R}\sum_{r=1}^{R}\left(\hat{\theta}_r(h)-\theta_r(h)\right)^2$

這些量可以幫助我們同時觀察 estimator 的系統性偏差與隨機波動。



In [1]:
import matplotlib
import numpy as np
import os
import time
import warnings

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tools.sm_exceptions import ConvergenceWarning


def kalman_filter(observations, process_variance=0.5, measurement_variance=10.0, initial_error=1.0):
    """Match the 1D Kalman smoothing procedure used in paper38."""
    initial_estimate = float(observations[0])
    estimated_state = initial_estimate
    estimate_error = float(initial_error)
    estimated_states = []
    for measurement in observations:
        predicted_state = estimated_state
        predicted_error = estimate_error + process_variance

        kalman_gain = predicted_error / (predicted_error + measurement_variance)
        estimated_state = predicted_state + kalman_gain * (measurement - predicted_state)
        estimate_error = (1.0 - kalman_gain) * predicted_error

        estimated_states.append(estimated_state)

    return np.array(estimated_states, dtype=float)


def decompose_with_simple_kf(series, process_variance=0.5, measurement_variance=10.0):
    estimated_low = kalman_filter(
        series,
        process_variance=process_variance,
        measurement_variance=measurement_variance,
        initial_error=1.0
    )
    estimated_high = np.array(series, dtype=float) - estimated_low
    return estimated_low, estimated_high


def create_windows(series, window_size):
    x_data = []
    y_data = []

    for idx in range(window_size, len(series)):
        x_data.append(series[idx - window_size:idx])
        y_data.append(series[idx])

    return np.array(x_data), np.array(y_data)


In [2]:
def recursive_ann_forecast(
    model,
    x_scaler,
    y_scaler,
    history,
    horizon,
    window_size,
    clip_bounds=None,
    innovation_sequence=None
):
    rolling = list(history[-window_size:])
    forecasts = []

    for step_idx in range(horizon):
        features = np.array(rolling[-window_size:], dtype=float).reshape(1, -1)
        features_scaled = x_scaler.transform(features)
        pred_scaled = model.predict(features_scaled).reshape(-1, 1)
        pred = y_scaler.inverse_transform(pred_scaled)[0, 0]
        if innovation_sequence is not None:
            pred = pred + float(innovation_sequence[step_idx])
        if clip_bounds is not None:
            # Prevent recursive forecasts from drifting far outside the training support.
            pred = float(np.clip(pred, clip_bounds[0], clip_bounds[1]))
        forecasts.append(pred)
        rolling.append(pred)

    return np.array(forecasts)

In [3]:
def find_arima_order(dataset, p_max=4, q_max=4):
    best_aic = np.inf
    best_order = (1, 0, 0)

    for p in range(p_max + 1):
        for q in range(q_max + 1):
            try:
                model = ARIMA(
                    dataset,
                    order=(p, 0, q),
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", ConvergenceWarning)
                    warnings.simplefilter("ignore", UserWarning)
                    result = model.fit()
                if result.aic < best_aic:
                    best_aic = result.aic
                    best_order = (p, 0, q)
            except Exception:
                continue

    return best_order


def fit_arima_low_component(low_train, horizon, arima_order):
    rolling_history = list(np.asarray(low_train, dtype=float))
    forecasts = []

    for _ in range(horizon):
        arima_model = ARIMA(
            rolling_history,
            order=arima_order,
            trend="c",
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            warnings.simplefilter("ignore", UserWarning)
            arima_result = arima_model.fit()
        pred = arima_result.get_forecast(1).predicted_mean[0]
        forecasts.append(pred)
        rolling_history.append(pred)

    return np.array(forecasts)


In [4]:
def fit_ann_model_bundle(high_train, window_size):
    x_train, y_train = create_windows(high_train, window_size)
    if len(x_train) == 0:
        raise ValueError("Not enough samples for ANN windows. Increase training size or reduce window_size.")

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()
    x_train_scaled = x_scaler.fit_transform(x_train)
    y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()

    ann_model = MLPRegressor(
        hidden_layer_sizes=(10, 10, 10),
        activation="relu",
        solver="adam",
        alpha=1e-5,
        learning_rate_init=0.001,
        max_iter=3000,
        random_state=42
    )
    ann_model.fit(x_train_scaled, y_train_scaled)

    # Use a robust empirical range from the training high component as a stability guard
    # for recursive multi-step ANN forecasts.
    lower_q, upper_q = np.quantile(high_train, [0.01, 0.99])
    iqr = np.subtract(*np.quantile(high_train, [0.75, 0.25]))
    margin = max(0.5 * iqr, 0.25)
    clip_bounds = (float(lower_q - margin), float(upper_q + margin))

    return {
        "model": ann_model,
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
        "clip_bounds": clip_bounds,
        "x_train_scaled": x_train_scaled,
        "y_train": y_train,
        "window_size": window_size,
    }


def fit_ann_high_component(high_train, horizon, window_size):
    ann_bundle = fit_ann_model_bundle(high_train, window_size)

    high_forecast = recursive_ann_forecast(
        ann_bundle["model"],
        ann_bundle["x_scaler"],
        ann_bundle["y_scaler"],
        high_train,
        horizon,
        window_size,
        clip_bounds=ann_bundle["clip_bounds"]
    )

    return high_forecast


In [5]:
def fit_hybrid_in_sample_residuals(train_series, arima_order, window_size):
    estimated_low, estimated_high = decompose_with_simple_kf(train_series)

    low_model = ARIMA(
        estimated_low,
        order=arima_order,
        trend="c",
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        warnings.simplefilter("ignore", UserWarning)
        low_result = low_model.fit()
    low_fitted = np.asarray(low_result.fittedvalues, dtype=float)

    ann_bundle = fit_ann_model_bundle(estimated_high, window_size)
    high_fitted_scaled = ann_bundle["model"].predict(ann_bundle["x_train_scaled"]).reshape(-1, 1)
    high_fitted = ann_bundle["y_scaler"].inverse_transform(high_fitted_scaled).ravel()

    combined_fitted = low_fitted[window_size:] + high_fitted
    actual_aligned = np.asarray(train_series, dtype=float)[window_size:]
    combined_residuals = actual_aligned - combined_fitted
    centered_combined_residuals = combined_residuals - np.mean(combined_residuals)
    high_residuals = ann_bundle["y_train"] - high_fitted
    centered_high_residuals = high_residuals - np.mean(high_residuals)

    return {
        "combined_residuals": np.asarray(combined_residuals, dtype=float),
        "centered_combined_residuals": np.asarray(centered_combined_residuals, dtype=float),
        "centered_high_residuals": np.asarray(centered_high_residuals, dtype=float),
        "ann_bundle": ann_bundle,
        "estimated_high": estimated_high,
    }


def hybrid_point_forecast(train_series, horizon, arima_order, window_size=30):
    estimated_low, estimated_high = decompose_with_simple_kf(train_series)
    low_forecast = fit_arima_low_component(estimated_low, horizon, arima_order=arima_order)
    high_forecast = fit_ann_high_component(estimated_high, horizon, window_size=window_size)

    return {
        "estimated_low": estimated_low,
        "estimated_high": estimated_high,
        "low_forecast": low_forecast,
        "high_forecast": high_forecast,
        "final_forecast": low_forecast + high_forecast,
    }


def insample_calibration_residuals(series, train_size, arima_order, window_size=30):
    train_series = np.asarray(series[:train_size], dtype=float)
    return fit_hybrid_in_sample_residuals(train_series, arima_order=arima_order, window_size=window_size)


def rolling_calibration_residuals(series, initial_train_size, calibration_end, block_size, arima_order, window_size=30):
    residuals = []

    for start in range(initial_train_size, calibration_end, block_size):
        stop = min(start + block_size, calibration_end)
        horizon = stop - start
        train_series = np.asarray(series[:start], dtype=float)
        actual_block = np.asarray(series[start:stop], dtype=float)

        forecast_bundle = hybrid_point_forecast(
            train_series,
            horizon,
            arima_order=arima_order,
            window_size=window_size
        )
        block_residuals = actual_block - forecast_bundle["final_forecast"]
        residuals.extend(block_residuals.tolist())

    return np.asarray(residuals, dtype=float)


def sort_bootstrap_residuals(residual_pool, horizon, rng):
    sampled_indices = rng.choice(len(residual_pool), size=horizon, replace=True)
    sampled_indices.sort()
    return residual_pool[sampled_indices]


def giordano_style_prediction_interval(
    point_forecast,
    calibration_residuals,
    horizon,
    alpha=0.05,
    bootstrap_replications=300,
    random_seed=123
):
    rng = np.random.default_rng(random_seed)
    bootstrap_paths = np.zeros((bootstrap_replications, horizon), dtype=float)

    for boot_idx in range(bootstrap_replications):
        sampled_residuals = sort_bootstrap_residuals(calibration_residuals, horizon, rng)
        bootstrap_paths[boot_idx, :] = point_forecast + sampled_residuals

    interval_lower = np.quantile(bootstrap_paths, alpha / 2, axis=0)
    interval_upper = np.quantile(bootstrap_paths, 1 - alpha / 2, axis=0)
    return interval_lower, interval_upper


def bootstrap_paths_for_interval(
    point_forecast,
    calibration_residuals,
    horizon,
    bootstrap_replications=300,
    random_seed=123
):
    rng = np.random.default_rng(random_seed)
    bootstrap_paths = np.zeros((bootstrap_replications, horizon), dtype=float)

    for boot_idx in range(bootstrap_replications):
        sampled_residuals = sort_bootstrap_residuals(calibration_residuals, horizon, rng)
        bootstrap_paths[boot_idx, :] = point_forecast + sampled_residuals

    return bootstrap_paths


def normal_bootstrap_interval(bootstrap_paths, z_value=1.959963984540054):
    path_mean = np.mean(bootstrap_paths, axis=0)
    path_std = np.std(bootstrap_paths, axis=0, ddof=1)
    interval_lower = path_mean - z_value * path_std
    interval_upper = path_mean + z_value * path_std
    return interval_lower, interval_upper


def summarize_estimator(forecasts, truths):
    forecasts = np.asarray(forecasts, dtype=float)
    truths = np.asarray(truths, dtype=float)
    errors = forecasts - truths
    bias = np.mean(errors, axis=0)
    variance = np.var(forecasts, axis=0, ddof=1)
    mse = np.mean(errors ** 2, axis=0)

    return {
        "bias_by_horizon": bias,
        "variance_by_horizon": variance,
        "mse_by_horizon": mse,
        "mean_abs_bias": float(np.mean(np.abs(bias))),
        "mean_variance": float(np.mean(variance)),
        "mean_mse": float(np.mean(mse)),
        "h1_bias": float(bias[0]),
        "h1_variance": float(variance[0]),
        "h1_mse": float(mse[0]),
        "errors_h1": errors[:, 0],
    }


def bootstrap_mean_test(sample, null_value=0.0, bootstrap_replications=2000, random_seed=321):
    sample = np.asarray(sample, dtype=float)
    observed_mean = float(np.mean(sample))
    centered_sample = sample - observed_mean + null_value
    rng = np.random.default_rng(random_seed)

    bootstrap_means = np.empty(bootstrap_replications, dtype=float)
    for idx in range(bootstrap_replications):
        draw = centered_sample[rng.choice(len(centered_sample), size=len(centered_sample), replace=True)]
        bootstrap_means[idx] = np.mean(draw)

    p_value = float(np.mean(np.abs(bootstrap_means - null_value) >= abs(observed_mean - null_value)))
    ci_draws = sample[rng.choice(len(sample), size=(bootstrap_replications, len(sample)), replace=True)].mean(axis=1)
    ci_lower, ci_upper = np.quantile(ci_draws, [0.025, 0.975])

    return {
        "observed_mean": observed_mean,
        "p_value": p_value,
        "ci_lower": float(ci_lower),
        "ci_upper": float(ci_upper),
    }

In [6]:
def simulate_babu_style_data(n_steps, noise_std=0.15):
    """Generate a Giordano-inspired synthetic series with linear M1 and nonlinear M9 components."""
    low_true = np.zeros(n_steps, dtype=float)
    high_true = np.zeros(n_steps, dtype=float)

    low_true[0] = np.random.normal(0, 0.2)
    high_true[0] = np.random.normal(0, 0.2)

    for idx in range(1, n_steps):
        low_true[idx] = 0.6 * low_true[idx - 1] + np.random.normal(0, 0.2)
        high_true[idx] = 0.8 * high_true[idx - 1] - 0.8 * high_true[idx - 1] / (1.0 + np.exp(-10.0 * high_true[idx - 1])) + np.random.normal(0, 0.2)

    noise = np.random.normal(0, noise_std, size=n_steps)
    measurements = low_true + high_true + noise
    return low_true, high_true, measurements


In [7]:
def run_single_experiment(
    train_size=300,
    horizon=1,
    window_size=15,
    bootstrap_replications=300,
    initial_train_size=None,
    calibration_block=None
):
    dt = 1.0
    n_steps = train_size + horizon
    t = np.arange(n_steps) * dt

    low_true, high_true, measurements = simulate_babu_style_data(n_steps)

    calibration_low, _ = decompose_with_simple_kf(measurements[:train_size])
    arima_order = find_arima_order(calibration_low, p_max=3, q_max=3)
    if initial_train_size is None:
        initial_train_size = max(window_size + 20, train_size // 2)
    if calibration_block is None:
        calibration_block = max(1, min(10, max(1, train_size - initial_train_size)))

    calibration_residuals = rolling_calibration_residuals(
        measurements,
        initial_train_size=initial_train_size,
        calibration_end=train_size,
        block_size=calibration_block,
        arima_order=arima_order,
        window_size=window_size
    )

    full_train_bundle = hybrid_point_forecast(
        measurements[:train_size],
        horizon,
        arima_order=arima_order,
        window_size=window_size
    )

    estimated_low_full, estimated_high_full = decompose_with_simple_kf(measurements)
    low_forecast = full_train_bundle["low_forecast"]
    high_forecast = full_train_bundle["high_forecast"]
    final_forecast = full_train_bundle["final_forecast"]

    bootstrap_paths = bootstrap_paths_for_interval(
        final_forecast,
        calibration_residuals,
        horizon,
        bootstrap_replications=bootstrap_replications
    )

    interval_lower, interval_upper = giordano_style_prediction_interval(
        final_forecast,
        calibration_residuals,
        horizon,
        alpha=0.05,
        bootstrap_replications=bootstrap_replications
    )
    normal_interval_lower, normal_interval_upper = normal_bootstrap_interval(bootstrap_paths)

    t_test = t[train_size:]
    measurements_test = measurements[train_size:]
    true_low_test = low_true[train_size:]
    true_high_test = high_true[train_size:]
    true_clean_test = true_low_test + true_high_test

    low_rmse = np.sqrt(np.mean((low_forecast - true_low_test) ** 2))
    high_rmse = np.sqrt(np.mean((high_forecast - true_high_test) ** 2))
    low_mse = np.mean((low_forecast - true_low_test) ** 2)
    high_mse = np.mean((high_forecast - true_high_test) ** 2)
    final_clean_rmse = np.sqrt(np.mean((final_forecast - true_clean_test) ** 2))
    final_clean_mse = np.mean((final_forecast - true_clean_test) ** 2)
    final_noisy_rmse = np.sqrt(np.mean((final_forecast - measurements_test) ** 2))
    final_noisy_mse = np.mean((final_forecast - measurements_test) ** 2)
    calibration_residual_mean = np.mean(calibration_residuals)
    calibration_residual_std = np.std(calibration_residuals, ddof=1)
    interval_widths = interval_upper - interval_lower
    mean_interval_width = np.mean(interval_widths)
    median_interval_width = np.median(interval_widths)
    normal_interval_widths = normal_interval_upper - normal_interval_lower
    normal_mean_interval_width = np.mean(normal_interval_widths)
    normal_median_interval_width = np.median(normal_interval_widths)
    clean_in_interval = (true_clean_test >= interval_lower) & (true_clean_test <= interval_upper)
    noisy_in_interval = (measurements_test >= interval_lower) & (measurements_test <= interval_upper)
    clean_coverage = np.mean(clean_in_interval) * 100
    noisy_coverage = np.mean(noisy_in_interval) * 100
    clean_in_normal_interval = (true_clean_test >= normal_interval_lower) & (true_clean_test <= normal_interval_upper)
    noisy_in_normal_interval = (measurements_test >= normal_interval_lower) & (measurements_test <= normal_interval_upper)
    normal_clean_coverage = np.mean(clean_in_normal_interval) * 100
    normal_noisy_coverage = np.mean(noisy_in_normal_interval) * 100

    return {
        "t": t,
        "t_test": t_test,
        "measurements": measurements,
        "measurements_test": measurements_test,
        "low_true": low_true,
        "high_true": high_true,
        "true_low_test": true_low_test,
        "true_high_test": true_high_test,
        "true_clean_test": true_clean_test,
        "estimated_low_full": estimated_low_full,
        "estimated_high_full": estimated_high_full,
        "low_forecast": low_forecast,
        "high_forecast": high_forecast,
        "final_forecast": final_forecast,
        "interval_lower": interval_lower,
        "interval_upper": interval_upper,
        "normal_interval_lower": normal_interval_lower,
        "normal_interval_upper": normal_interval_upper,
        "clean_in_interval": clean_in_interval,
        "noisy_in_interval": noisy_in_interval,
        "clean_in_normal_interval": clean_in_normal_interval,
        "noisy_in_normal_interval": noisy_in_normal_interval,
        "arima_order": arima_order,
        "calibration_residuals": calibration_residuals,
        "calibration_residual_mean": calibration_residual_mean,
        "calibration_residual_std": calibration_residual_std,
        "mean_interval_width": mean_interval_width,
        "median_interval_width": median_interval_width,
        "normal_mean_interval_width": normal_mean_interval_width,
        "normal_median_interval_width": normal_median_interval_width,
        "low_rmse": low_rmse,
        "high_rmse": high_rmse,
        "low_mse": low_mse,
        "high_mse": high_mse,
        "final_clean_rmse": final_clean_rmse,
        "final_clean_mse": final_clean_mse,
        "final_noisy_rmse": final_noisy_rmse,
        "final_noisy_mse": final_noisy_mse,
        "clean_coverage": clean_coverage,
        "noisy_coverage": noisy_coverage,
        "normal_clean_coverage": normal_clean_coverage,
        "normal_noisy_coverage": normal_noisy_coverage,
    }


if __name__ == "__main__":
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    warnings.filterwarnings("ignore", category=UserWarning)

    np.random.seed(42)
    start_time = time.perf_counter()
    n_monte_carlo = 50
    train_size = 100
    horizons = [1 ,2 ,3]
    bootstrap_replications = 50

    print(f"Monte Carlo runs               : {n_monte_carlo}")
    print(f"Observed sample size T         : {train_size}")
    print(f"Bootstrap replications B       : {bootstrap_replications}")
    print("")
    print("h | Mean final MSE | Percentile coverage | Normal coverage | Percentile width | Normal width")
    print("--|----------------|---------------------|-----------------|------------------|-------------")

    for horizon in horizons:
        experiment_results = []
        for run_idx in range(n_monte_carlo):
            np.random.seed(42 + run_idx)
            experiment_results.append(
                run_single_experiment(
                    train_size=train_size,
                    horizon=horizon,
                    bootstrap_replications=bootstrap_replications
                )
            )

        clean_mses = np.array([result["final_clean_mse"] for result in experiment_results], dtype=float)
        clean_coverages = np.array([result["clean_coverage"] for result in experiment_results], dtype=float)
        normal_clean_coverages = np.array([result["normal_clean_coverage"] for result in experiment_results], dtype=float)
        mean_interval_widths = np.array([result["mean_interval_width"] for result in experiment_results], dtype=float)
        normal_mean_interval_widths = np.array([result["normal_mean_interval_width"] for result in experiment_results], dtype=float)

        final_forecasts = np.array([result["final_forecast"] for result in experiment_results], dtype=float)
        final_truths = np.array([result["true_clean_test"] for result in experiment_results], dtype=float)
        estimator_summary = summarize_estimator(final_forecasts, final_truths)
        mean_error_test = bootstrap_mean_test(estimator_summary["errors_h1"], null_value=0.0)

        print(
            f"{horizon} | "
            f"{np.mean(clean_mses):.4f}         | "
            f"{np.mean(clean_coverages):.2f}%                | "
            f"{np.mean(normal_clean_coverages):.2f}%            | "
            f"{np.mean(mean_interval_widths):.4f}           | "
            f"{np.mean(normal_mean_interval_widths):.4f}"
        )

        print("")
        print("Final forecast estimator summary")
        print(f"Mean absolute bias              : {estimator_summary['mean_abs_bias']:.4f}")
        print(f"Mean variance                  : {estimator_summary['mean_variance']:.4f}")
        print(f"Mean MSE                       : {estimator_summary['mean_mse']:.4f}")
        print(f"Horizon 1 bias                 : {estimator_summary['h1_bias']:.4f}")
        print(f"Horizon 1 variance             : {estimator_summary['h1_variance']:.4f}")
        print(f"Horizon 1 MSE                  : {estimator_summary['h1_mse']:.4f}")
        print("")
        print("Bootstrap CI comparison")
        print(f"Percentile mean width          : {np.mean(mean_interval_widths):.4f}")
        print(f"Normal mean width              : {np.mean(normal_mean_interval_widths):.4f}")
        print(f"Percentile mean clean coverage : {np.mean(clean_coverages):.2f}%")
        print(f"Normal mean clean coverage     : {np.mean(normal_clean_coverages):.2f}%")
        print("")
        print("Bootstrap hypothesis test on horizon-1 forecast error")
        print("H0: E[e1] = 0")
        print(f"Observed mean error            : {mean_error_test['observed_mean']:.4f}")
        print(f"Bootstrap p-value              : {mean_error_test['p_value']:.4f}")
        print(f"95% bootstrap CI               : [{mean_error_test['ci_lower']:.4f}, {mean_error_test['ci_upper']:.4f}]")

    elapsed_seconds = time.perf_counter() - start_time
    print("")
    print(f"Elapsed time                   : {elapsed_seconds:.2f} seconds")


Monte Carlo runs               : 50
Observed sample size T         : 100
Bootstrap replications B       : 50

h | Mean final MSE | Percentile coverage | Normal coverage | Percentile width | Normal width
--|----------------|---------------------|-----------------|------------------|-------------
1 | 0.2168         | 92.00%                | 96.00%            | 1.6553           | 1.7984

Final forecast estimator summary
Mean absolute bias              : 0.0550
Mean variance                  : 0.1556
Mean MSE                       : 0.2168
Horizon 1 bias                 : -0.0550
Horizon 1 variance             : 0.1556
Horizon 1 MSE                  : 0.2168

Bootstrap CI comparison
Percentile mean width          : 1.6553
Normal mean width              : 1.7984
Percentile mean clean coverage : 92.00%
Normal mean clean coverage     : 96.00%

Bootstrap hypothesis test on horizon-1 forecast error
H0: E[e1] = 0
Observed mean error            : -0.0550
Bootstrap p-value              : 0.3980
95

## 五、模擬結果

本作業利用 Monte Carlo simulation 重複進行資料生成、模型建構與預測流程，並在不同 forecast horizon 下評估 hybrid forecasting model 的表現。主要報告的結果包括 final forecast estimator 的 bias、variance 與 MSE，以及 bootstrap confidence interval 的 coverage、interval width，還有 bootstrap hypothesis testing 的結果。

在 horizon $h = 1$ 的情況下，final forecast 的 mean MSE 為 **0.2168**，mean absolute bias 為 **0.0550**，mean variance 為 **0.1556**。對應的 horizon 1 bias 為 **-0.0550**，variance 為 **0.1556**，MSE 為 **0.2168**。在 bootstrap confidence interval 的表現上，percentile interval 的平均 coverage 為 **92.00%**，normal interval 的平均 coverage 為 **96.00%**；percentile interval 的平均寬度為 **1.6553**，normal interval 的平均寬度為 **1.7984**。在 hypothesis testing 的部分，observed mean forecast error 為 **-0.0550**，bootstrap p-value 為 **0.3980**，95% bootstrap confidence interval 為 **[-0.1877, 0.0639]**。

在 horizon $h = 2$ 的情況下，final forecast 的 mean MSE 為 **0.1518**，mean absolute bias 為 **0.0702**，mean variance 為 **0.0841**。對應的 horizon 1 bias 為 **0.0267**，variance 為 **0.1198**，MSE 為 **0.1307**。bootstrap confidence interval 的結果顯示，percentile interval 的平均 coverage 為 **91.00%**，normal interval 的平均 coverage 為 **95.00%**；percentile interval 的平均寬度為 **1.6149**，normal interval 的平均寬度為 **1.7932**。在 hypothesis testing 上，observed mean forecast error 為 **0.0267**，bootstrap p-value 為 **0.6025**，95% bootstrap confidence interval 為 **[-0.0706, 0.1264]**。

在 horizon $h = 3$ 的情況下，final forecast 的 mean MSE 為 **0.2110**，mean absolute bias 為 **0.0826**，mean variance 為 **0.1324**。對應的 horizon 1 bias 為 **0.0278**，variance 為 **0.1730**，MSE 為 **0.1827**。在 confidence interval 的表現上，percentile interval 的平均 coverage 為 **89.33%**，normal interval 的平均 coverage 為 **94.67%**；percentile interval 的平均寬度為 **1.5711**，normal interval 的平均寬度為 **1.7504**。在 hypothesis testing 的部分，observed mean forecast error 為 **0.0278**，bootstrap p-value 為 **0.6515**，95% bootstrap confidence interval 為 **[-0.0910, 0.1438]**。

綜合三組 horizon 的結果，可以發現 final forecast estimator 的 mean absolute bias 都維持在相對小的範圍內，表示模型平均而言沒有明顯的系統性偏誤。另一方面，bootstrap confidence interval 的 empirical coverage 已經相當接近 nominal 95% 的信心水準，尤其 normal interval 的 coverage 最接近 95%，但其 interval width 也 consistently 較大，表示 normal interval 較為保守。相比之下，percentile interval 的區間較窄，但 coverage 略低。

在 bootstrap hypothesis testing 中，三組 horizon 的 p-value 均大於 0.05，且對應的 95% bootstrap confidence intervals 都包含 0，表示目前沒有足夠證據認為一步預測誤差的平均值顯著不等於 0。


## 六、結論

本作業利用 Kalman filter 分解、ARIMA 與 ANN 所組成的 hybrid forecasting model，並結合 Monte Carlo simulation 與 bootstrap methods，對模型的預測表現與不確定性進行評估。相較於單純只看點預測誤差，本研究進一步分析了 estimator 的 bias、variance、MSE，以及 bootstrap confidence interval 與 hypothesis testing 的結果，因此能更完整地描述模型的統計性質。

- estimator 本身的表現而言，final forecast 的 mean absolute bias 在不同 horizon 下都維持在相對小的範圍內，表示此 hybrid estimator 平均而言沒有非常明顯的系統性偏差。同時，variance 與 MSE 的結果也顯示，模型在重複抽樣下雖然仍存在一定程度的波動，但整體表現仍具有可接受的穩定性。這代表透過 Kalman filter 先將原始混合訊號分解，再分別以 ARIMA 與 ANN 建模，確實可以有效整合低頻線性資訊與高頻非線性資訊。

- 在 confidence interval 的建構上，採用 **rolling out-of-sample residuals** 作為 bootstrap residual pool。主要是在訓練資料區間內使用 rolling window 的方式，反覆建立多個 out-of-sample prediction blocks。對每一個 block，都以前段樣本建立 hybrid forecast model，並將實際觀測值與預測值的差視為 out-of-sample residuals。最後，再從這些 rolling residuals 中進行 resampling with replacement，建立 bootstrap forecast paths，進一步構造 confidence intervals。

另一方面，normal interval 的平均寬度 consistently 大於 percentile interval，因此 normal interval 雖然 coverage 較高，但也較為保守；percentile interval 較窄，但 coverage 稍低。這反映出 confidence interval 建構中常見的 trade-off：較寬的區間通常較容易達到較高 coverage，但也可能降低區間推論的精確性；較窄的區間則提供較集中的估計，但 coverage 可能略低於 nominal level。因此，在本作業的結果中，可以將 normal interval 視為較保守的選擇，而 percentile interval 則可視為較精簡但略低估不確定性的做法。

在 hypothesis testing 的部分，本作業針對 horizon 1 的平均 forecast error 進行 bootstrap hypothesis testing。檢定的虛無假設為：

$H_0: E[e_1] = 0$

其中 $e_1$ 表示一步預測誤差。這個檢定的目的，是要判斷模型在一步預測上是否存在顯著的系統性偏誤。從本次模擬結果來看，各組設定下的 bootstrap p-value 都大於 0.05，且對應的 95% bootstrap confidence interval 均包含 0。這表示目前沒有足夠證據拒絕虛無假設，表示在 horizon 1 下，模型的平均預測誤差並沒有顯著偏離 0。從統計推論的角度來看，這支持了此 hybrid model 在一步預測上沒有明顯系統性 bias。


本作業顯示 resampling methods 不只是輔助工具，而是評估 forecasting model 的重要方法。透過 bootstrap confidence intervals 與 hypothesis testing，可以更完整地理解預測模型的可靠度與不確定性。相較於只報告單一誤差指標，這樣的分析更能呈現模型在重複抽樣下的真實表現，也更符合統計推論的精神。


## 七、未來展望

本作業雖然已經利用 Monte Carlo simulation 與 bootstrap methods 對 hybrid forecasting model 的統計性質進行分析，但仍有幾個值得進一步延伸的方向。

1. 可以比較更多不同形式的 bootstrap confidence interval，例如 BCa interval 或 studentized bootstrap interval，以檢查在相同資料生成機制下，這些方法是否能提供更穩定或更準確的 coverage 表現。由於本作業目前主要比較 percentile interval 與 normal approximation interval，因此未來若加入更多區間建構方法，將能更完整地理解不同 bootstrap procedures 的差異。

2. 可以進一步研究樣本數與 bootstrap replication 數量對結果的影響。例如，當訓練樣本數較小時，estimator 的 bias、variance 與 MSE 可能會有不同的變化；同樣地，bootstrap replications 的數量若提高，也可能使 confidence interval 與 hypothesis testing 的結果更穩定。因此，未來可以系統性地探討 sample size 與 replication size 在此問題中的角色。

3. 可以將本作業的 hybrid forecasting framework 與其他 benchmark models 進行比較，例如單純的 ARIMA model、單純的 ANN model，或其他常見的 time series forecasting methods。透過這樣的比較，將能更清楚地判斷 Kalman filter decomposition 結合 ARIMA 與 ANN 的混合模型，是否在預測精度與統計推論上具有相對優勢。


## 八、參考資料

1. C. N. Babu and B. E. Reddy, “A moving-average filter based hybrid arima–ann model for forecasting time series data,” Applied Soft Computing,vol. 23, pp. 27–38, 2014.
2. F. Giordano, M. La Rocca, and C. Perna, “Forecasting nonlinear time series with neural network sieve bootstrap,” Computational Statistics & Data Analysis, vol. 51, no. 8, pp. 3871–3884, 2007.
3. Hamilton, J. D. (1994). *Time Series Analysis*. Princeton University Press.
4. G. P. Zhang, “Time series forecasting using a hybrid arima and neural network model,” Neurocomputing, vol. 50, pp. 159–175, 2003.


## 九、環境

本作業使用的執行環境如下：

- **Python version**: 3.14.3  
- **Platform**: Windows 10 (10.0.19045-SP0)  
- **NumPy**: 2.4.2  
- **Matplotlib**: 3.10.8  
- **scikit-learn**: 1.8.0  
- **statsmodels**: 0.14.6  

以上套件主要用於模擬資料生成、Kalman filter 分解、ARIMA 與 ANN 模型建構，以及 bootstrap resampling 與結果視覺化。
